# DXA prototype v5 — компактная версия

Цель: оставить только рабочие части прототипа и шесть проверок из схемы:

**Бедро**
1. позиция — три узких места;
2. ротация — размер/угол «горбика»;
3. металл — отличие скрытого представления.

**Позвоночник**
4. углы — ось позвоночника относительно вертикали;
5. позиция — доля площади подвздошных костей;
6. проблемы — разность углов первого и последнего позвонков как прототип показателя сколиоза.

Изображения берутся только из `Исследования`, итоговая анатомическая разметка — только из `Размеченные/labels.csv`.

## 0. Установка

In [ ]:
%pip install -q pydicom pylibjpeg pylibjpeg-libjpeg numpy pandas matplotlib scipy scikit-image scikit-learn pillow tqdm torch torchvision opencv-python-headless openpyxl
# Для пункта с подвздошными костями используется pretrained Segment Anything (SAM).
# Если пакет ещё не установлен:
# %pip install -q https://github.com/facebookresearch/segment-anything/archive/refs/heads/main.zip

## 1. Импорты и конфигурация

In [ ]:
from pathlib import Path, PurePosixPath
from contextlib import nullcontext
import math, random, re, urllib.request, warnings

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pydicom
from PIL import Image
from scipy.ndimage import gaussian_filter1d, map_coordinates
from scipy.signal import find_peaks, savgol_filter
from sklearn.decomposition import PCA
from sklearn.metrics import balanced_accuracy_score, f1_score, classification_report, confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.models import resnet18, ResNet18_Weights

try:
    from pydicom.pixels import apply_modality_lut, apply_voi_lut
except ImportError:
    from pydicom.pixel_data_handlers.util import apply_modality_lut, apply_voi_lut

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def find_project_root(start=Path.cwd()):
    start = start.resolve()

    for path in [start, *start.parents]:
        if (path / "labeler").is_dir() and (path / "prototype").is_dir():
            return path

    raise FileNotFoundError(
        "Не удалось определить корень проекта. "
        "Ожидаются папки 'labeler' и 'prototype'."
    )


DATASET_ROOT = find_project_root()

STUDIES_ROOT = DATASET_ROOT / "Исследования"
LABELS_CSV = DATASET_ROOT / "Размеченные" / "labels.csv"
TASK_TABLE_CANDIDATES = [
    DATASET_ROOT / "разметка.xlsx",
    DATASET_ROOT / "разметка.csv",
]
TASK_TABLE_PATH = next(
    (p for p in TASK_TABLE_CANDIDATES if p.exists()),
    None,
)

WORK_DIR = DATASET_ROOT / "prototype" / "output"
WORK_DIR.mkdir(parents=True, exist_ok=True)

SAM_CHECKPOINT = (
    DATASET_ROOT
    / "prototype"
    / "models"
    / "sam_vit_b_01ec64.pth"
)
SAM_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
STUDIES_ROOT = DATASET_ROOT / "Исследования"
LABELS_CSV = DATASET_ROOT / "Размеченные" / "labels.csv"
TASK_TABLE_CANDIDATES = [
    DATASET_ROOT / "разметка.xlsx",
    DATASET_ROOT / "разметка.csv",
]
TASK_TABLE_PATH = next((p for p in TASK_TABLE_CANDIDATES if p.exists()), None)
WORK_DIR = DATASET_ROOT / "prototype_output"
WORK_DIR.mkdir(exist_ok=True)

EXPECTED_COUNTS = {"LEG": 333, "SPINE": 166}
BACKGROUND_QUANTILE = 0.01
IMAGE_SIZE, BATCH_SIZE, NUM_WORKERS = 224, 16, 0
HEAD_EPOCHS, FINETUNE_EPOCHS = 3, 5

# Размеры графиков примерно вдвое меньше прежних по каждой оси.
FIG_SMALL = (4, 3)
FIG_WIDE = (7, 3.5)
FIG_TALL = (4, 5)

# Бедро: линии заданы относительно вертикали.
HIP_ANGLES = {"1": -30.0, "2": 45.0, "3": -30.0}
HIP_OFFSET_SEARCH_PX, HIP_LINE_HALF_LENGTH_PX = 45, 160
HIP_MIN_NONZERO_PIXELS = 5

# Позвоночник.
SPINE_N_VERTEBRAE = 7
SPINE_N_SEPARATORS = SPINE_N_VERTEBRAE - 1
SPINE_BRIGHT_FRACTION = 0.10
SPINE_MIN_SEPARATOR_DISTANCE_DIVISOR = 7
SPINE_SEGMENT_NAMES = ["T12", "L1", "L2", "L3", "L4", "L5", "SACRUM"]

# SAM.
SAM_MODEL_TYPE = "vit_b"
SAM_CHECKPOINT = DATASET_ROOT / "sam_vit_b_01ec64.pth"
SAM_CHECKPOINT_URL = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"

print("DEVICE:", DEVICE)
print("DATASET_ROOT:", DATASET_ROOT)

## 2. DICOM, мягкое отсечение фона и финальная разметка

In [ ]:
def normalize_relpath(value):
    return str(value or "").strip().replace("\\", "/").lstrip("./")

def rel_to_path(value):
    return Path(*PurePosixPath(normalize_relpath(value)).parts)

def read_dicom_image(path):
    ds = pydicom.dcmread(path, force=True)
    arr = np.asarray(apply_modality_lut(ds.pixel_array, ds), dtype=np.float32)
    try: arr = np.asarray(apply_voi_lut(arr, ds), dtype=np.float32)
    except Exception: pass
    finite = np.isfinite(arr)
    lo, hi = np.nanpercentile(arr[finite], [0.5, 99.5])
    if hi <= lo: lo, hi = np.nanmin(arr[finite]), np.nanmax(arr[finite])
    img = np.clip((arr - lo) / max(hi - lo, 1e-6), 0, 1)
    if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1": img = 1 - img
    return ds, img.astype(np.float32)

def threshold_background(img, q=BACKGROUND_QUANTILE):
    positive = img[img > 0]
    if not len(positive): return img.copy(), 0.0
    threshold = float(np.quantile(positive, q))
    out = img.copy()
    out[out < threshold] = 0
    return out, threshold

def preprocess_dicom(path):
    ds, img = read_dicom_image(path)
    filtered, threshold = threshold_background(img)
    return ds, img, filtered, threshold

def load_labels():
    df = pd.read_csv(LABELS_CSV, dtype=str).fillna("")
    df["relative_path"] = df["relative_path"].map(normalize_relpath)
    df["label"] = df["label"].str.strip().str.upper()
    if "side" not in df.columns: df["side"] = ""
    df["side"] = df["side"].str.strip().str.upper()
    if "metal" not in df.columns: df["metal"] = ""
    df["metal"] = df["metal"].astype(str).str.strip()

    df["dicom_path"] = df["relative_path"].map(lambda p: STUDIES_ROOT / rel_to_path(p))
    missing = df[~df["dicom_path"].map(Path.exists)]
    if len(missing): raise FileNotFoundError(f"Не найдено {len(missing)} DICOM из labels.csv")

    counts = df["label"].value_counts().to_dict()
    if len(df) != 499 or any(counts.get(k, 0) != v for k, v in EXPECTED_COUNTS.items()):
        raise ValueError(f"Неожиданная разметка: n={len(df)}, counts={counts}")

    df["study_id"] = df["relative_path"].str.split("/").str[0]
    df["anatomy3"] = np.where(
        df["label"].eq("SPINE"), "SPINE",
        np.where(df["side"].eq("LEFT"), "LEG_LEFT",
                 np.where(df["side"].eq("RIGHT"), "LEG_RIGHT", ""))
    )
    return df

labels_df = load_labels()
display(labels_df["label"].value_counts().rename("count").to_frame())
print("Нога с указанной стороной:", (labels_df["anatomy3"] != "").sum() - (labels_df["label"] == "SPINE").sum())
print("Металл размечен у ног:", labels_df.loc[labels_df["label"].eq("LEG"), "metal"].isin(["0", "1"]).sum(), "/", (labels_df["label"] == "LEG").sum())

### Левая / правая нога

`labels.csv` теперь может содержать дополнительную колонку `side`:

- `LEFT`;
- `RIGHT`;
- пусто — сторона ещё не размечена.

До завершения этой разметки старый `LEG/SPINE` остаётся валидным. Трёхклассовая модель использует только строки, где сторона ноги известна.

## 3. Компактный ResNet-классификатор и скрытые представления

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
transform = T.Compose([T.Resize((IMAGE_SIZE, IMAGE_SIZE)), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

def dicom_to_pil(path):
    _, img, _, _ = preprocess_dicom(path)
    return Image.fromarray((img * 255).astype(np.uint8)).convert("RGB")

class DicomDataset(Dataset):
    def __init__(self, df, class_col, class_to_idx):
        self.df, self.class_col, self.class_to_idx = df.reset_index(drop=True), class_col, class_to_idx
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        return transform(dicom_to_pil(row["dicom_path"])), torch.tensor(self.class_to_idx[row[self.class_col]], dtype=torch.long), str(row["dicom_path"])

def build_resnet(n_classes):
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    for p in model.parameters(): p.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, n_classes)
    return model.to(DEVICE)

def amp_ctx():
    return torch.amp.autocast(device_type="cuda") if DEVICE.type == "cuda" else nullcontext()

def run_epoch(model, loader, loss_fn, optimizer=None):
    train = optimizer is not None
    model.train(train)
    losses, yt, yp = [], [], []
    with torch.set_grad_enabled(train):
        for x, y, _ in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            if train: optimizer.zero_grad(set_to_none=True)
            with amp_ctx():
                logits = model(x)
                loss = loss_fn(logits, y)
            if train: loss.backward(); optimizer.step()
            losses.append(loss.item()); yt.extend(y.cpu().tolist()); yp.extend(logits.argmax(1).detach().cpu().tolist())
    return {"loss": np.mean(losses), "balanced_acc": balanced_accuracy_score(yt, yp), "f1_macro": f1_score(yt, yp, average="macro", zero_division=0)}

def fit_all_data_classifier(df, class_col, epochs_head=HEAD_EPOCHS, epochs_ft=FINETUNE_EPOCHS):
    classes = sorted(df[class_col].unique())
    class_to_idx = {c: i for i, c in enumerate(classes)}
    loader = DataLoader(DicomDataset(df, class_col, class_to_idx), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    eval_loader = DataLoader(DicomDataset(df, class_col, class_to_idx), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    counts = df[class_col].value_counts()
    weights = torch.tensor([len(df) / (len(classes) * counts[c]) for c in classes], dtype=torch.float32, device=DEVICE)
    model, loss_fn = build_resnet(len(classes)), nn.CrossEntropyLoss(weight=weights)
    history = []

    opt = torch.optim.AdamW(model.fc.parameters(), lr=1e-3)
    for e in range(epochs_head):
        tr = run_epoch(model, loader, loss_fn, opt); ev = run_epoch(model, eval_loader, loss_fn)
        history.append({"phase": "head", "epoch": e + 1, **{f"train_{k}": v for k, v in tr.items()}, **{f"all_{k}": v for k, v in ev.items()}})

    for p in model.layer4.parameters(): p.requires_grad = True
    opt = torch.optim.AdamW([{"params": model.layer4.parameters(), "lr": 1e-4}, {"params": model.fc.parameters(), "lr": 3e-4}])
    for e in range(epochs_ft):
        tr = run_epoch(model, loader, loss_fn, opt); ev = run_epoch(model, eval_loader, loss_fn)
        history.append({"phase": "finetune", "epoch": e + 1, **{f"train_{k}": v for k, v in tr.items()}, **{f"all_{k}": v for k, v in ev.items()}})
    return model, class_to_idx, pd.DataFrame(history)

anatomy_model = None
three_class_df = labels_df[labels_df["anatomy3"].ne("")].copy()
print("Доступно для 3-классовой модели:", three_class_df["anatomy3"].value_counts().to_dict())
print("Запуск обучения: anatomy_model, anatomy_classes, history = fit_all_data_classifier(three_class_df, 'anatomy3')")

In [ ]:
@torch.no_grad()
def pretrained_embedding(path):
    model = resnet18(weights=ResNet18_Weights.DEFAULT).to(DEVICE).eval()
    encoder = nn.Sequential(*list(model.children())[:-1])
    x = transform(dicom_to_pil(path)).unsqueeze(0).to(DEVICE)
    return encoder(x).flatten(1)[0].cpu().numpy()

def show_classification_examples(model, df, class_col, class_to_idx, n=6):
    idx_to_class = {v: k for k, v in class_to_idx.items()}
    sample = df.sample(min(n, len(df)), random_state=SEED)
    fig, axes = plt.subplots(2, 3, figsize=(6, 4))
    for ax in axes.flat: ax.axis("off")
    model.eval()
    with torch.no_grad():
        for ax, (_, row) in zip(axes.flat, sample.iterrows()):
            x = transform(dicom_to_pil(row["dicom_path"])).unsqueeze(0).to(DEVICE)
            pred = idx_to_class[int(model(x).argmax(1))]
            _, img, _, _ = preprocess_dicom(row["dicom_path"])
            ax.imshow(img, cmap="gray"); ax.set_title(f"{row[class_col]} → {pred}", fontsize=8)
    plt.tight_layout(); plt.show()

## 4. `разметка.csv`: извлечение разметки шести задач

In [ ]:
UID_RE = re.compile(r"^(?:2\.25\.\d+|1\.2(?:\.\d+)+)$")

def read_task_raw(path):
    """Читает разметка.xlsx или разметка.csv без фиксированного числа строк заголовка."""
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path, header=None, dtype=str).fillna("")

    last_error = None
    for enc in ("utf-8-sig", "cp1251", "utf-8"):
        try:
            return pd.read_csv(path, header=None, sep=None, engine="python", dtype=str, encoding=enc).fillna("")
        except Exception as e:
            last_error = e
    raise last_error


def load_task_table(path=TASK_TABLE_PATH):
    if path is None:
        print("Файл клинической разметки не найден.")
        print("Ожидался один из файлов:")
        for p in TASK_TABLE_CANDIDATES:
            print(" -", p)
        return pd.DataFrame(), {}

    print("Используется таблица задач:", path)
    raw = read_task_raw(path)
    print("Размер исходной таблицы:", raw.shape)

    # Ищем колонку study по значениям UID, а не по имени/номеру столбца.
    uid_counts = {
        c: raw[c].astype(str).str.strip().map(lambda x: bool(UID_RE.match(x))).sum()
        for c in raw.columns
    }
    study_col = max(uid_counts, key=uid_counts.get)
    if uid_counts[study_col] == 0:
        raise ValueError("В таблице не найден столбец со study UID.")

    is_data = raw[study_col].astype(str).str.strip().map(lambda x: bool(UID_RE.match(x)))
    first_data = int(np.flatnonzero(is_data.to_numpy())[0])

    # Все строки над первой строкой study считаем многоуровневым заголовком.
    headers = raw.iloc[:first_data].astype(str)
    top = headers.iloc[0].replace("", np.nan).ffill().fillna("") if len(headers) else pd.Series("", index=raw.columns)

    names = {}
    for c in raw.columns:
        tokens = [x.strip() for x in headers[c].tolist() if x.strip()]
        if top[c] and (not tokens or top[c] != tokens[0]):
            tokens.insert(0, top[c])
        names[c] = " | ".join(dict.fromkeys(tokens)) or f"col_{c}"

    data = raw.loc[is_data].copy()
    data.columns = [names[c] for c in raw.columns]
    data = data.rename(columns={names[study_col]: "study"})
    data["study"] = data["study"].astype(str).str.strip()

    def norm_header(text):
        return str(text).lower().replace("ё", "е").replace("\xa0", " ")

    def find_col(*groups):
        for col in data.columns:
            text = norm_header(col)
            if all(any(key in text for key in group) for group in groups):
                return col
        return None

    mapping = {
        "spine_alignment": find_col(("позвоноч",), ("выравн", "ось")),
        "spine_artifacts": find_col(("позвоноч",), ("посторон", "артеф", "налож")),
        "right_hip_position_rotation": find_col(("прав",), ("бедр",), ("позиционир", "ротац")),
        "left_hip_position_rotation": find_col(("лев",), ("бедр",), ("позиционир", "ротац")),
        "right_hip_roi": find_col(("прав",), ("бедр",), ("област", "интерес")),
        "left_hip_roi": find_col(("лев",), ("бедр",), ("област", "интерес")),
        "hip_metal": find_col(("металл",)),
        "spine_position": find_col(("подвздош",)),
        "comment": find_col(("комментар",)),
    }

    out = pd.DataFrame({"study": data["study"]})
    for task, col in mapping.items():
        if col is not None:
            out[task] = data[col].astype(str).str.strip()

    # Собираем комментарии из всех колонок, в чьем заголовке есть "комментар".
    comment_cols = [c for c in data.columns if "комментар" in norm_header(c)]
    if comment_cols:
        comments = data[comment_cols].astype(str).agg(" | ".join, axis=1)
        out["comment_all"] = comments
        out["scoliosis_from_comment"] = comments.str.contains("сколиоз", case=False, na=False).astype(int)

    print("\nАвтоматически найденные соответствия:")
    display(pd.DataFrame({"task": list(mapping), "column": [mapping[k] for k in mapping]}))
    print(f"\nРаспознано строк с study UID: {len(out)}")
    display(out.head(8))
    return out.reset_index(drop=True), mapping


task_labels, task_mapping = load_task_table()

По присланной структуре файла отдельные метки **позиции** и **ротации** бедра объединены в одном поле `позиционирование/ротация`. Поэтому код сохраняет этот общий target и не придумывает отсутствующее разделение. Для металла и положения подвздошных костей используются соответствующие столбцы только если они реально обнаружены.

## 5. Бедро — позиция: три узких места

In [ ]:
def line_direction(angle_from_vertical):
    a = np.deg2rad(angle_from_vertical)
    return np.array([np.sin(a), np.cos(a)], dtype=np.float32)

def evaluate_line(img, seed_xy, angle, offset):
    d = line_direction(angle); p = np.array([d[1], -d[0]])
    center = np.asarray(seed_xy, dtype=float) + offset * p
    t = np.arange(-HIP_LINE_HALF_LENGTH_PX, HIP_LINE_HALF_LENGTH_PX + 1)
    xs = np.rint(center[0] + t * d[0]).astype(int); ys = np.rint(center[1] + t * d[1]).astype(int)
    valid = (xs >= 0) & (xs < img.shape[1]) & (ys >= 0) & (ys < img.shape[0])
    xs, ys, t = xs[valid], ys[valid], t[valid]
    values = img[ys, xs]; nz = values > 0
    width = int(nz.sum())
    if width < HIP_MIN_NONZERO_PIXELS: return None
    tn, vn = t[nz], values[nz]
    com_t = float(np.sum(tn * vn) / max(vn.sum(), 1e-6))
    return {"width": width, "center": center, "com": center + com_t * d,
            "p1": center + tn.min() * d, "p2": center + tn.max() * d, "offset": offset}

def narrowest_line(img, seed_xy, angle):
    candidates = [evaluate_line(img, seed_xy, angle, o) for o in range(-HIP_OFFSET_SEARCH_PX, HIP_OFFSET_SEARCH_PX + 1)]
    candidates = [c for c in candidates if c is not None]
    return min(candidates, key=lambda c: (c["width"], abs(c["offset"]))) if candidates else None

def select_points(path, labels=("1", "2", "3")):
    _, img, _, _ = preprocess_dicom(path)
    fig, ax = plt.subplots(figsize=FIG_TALL); ax.imshow(img, cmap="gray"); ax.set_title("Кликните: " + ", ".join(labels)); ax.axis("off")
    points = plt.ginput(len(labels), timeout=-1, show_clicks=True); plt.close(fig)
    return {label: tuple(map(float, p)) for label, p in zip(labels, points)}

def analyze_hip_position(path, points):
    _, img, filtered, _ = preprocess_dicom(path)
    results = {k: narrowest_line(filtered, points[k], HIP_ANGLES[k]) for k in ("1", "2", "3")}
    fig, ax = plt.subplots(figsize=FIG_TALL); ax.imshow(img, cmap="gray"); ax.axis("off")
    rows = []
    for k, r in results.items():
        if r is None: continue
        ax.plot([r["p1"][0], r["p2"][0]], [r["p1"][1], r["p2"][1]], lw=2)
        ax.scatter(*zip(r["com"]), s=20); ax.text(r["com"][0] + 3, r["com"][1], f"{k}: {r['width']} px", fontsize=7)
        rows.append({"structure": k, "angle_from_vertical": HIP_ANGLES[k], "width_nonzero_px": r["width"], "offset_px": r["offset"]})
    ax.set_title("3 узких места"); plt.tight_layout(); plt.show()
    return pd.DataFrame(rows)

## 6. Бедро — ротация: «горбик» без threshold

In [ ]:
def trace_edge(img, seed_xy, half_w=90, half_h=120, continuity=20):
    h, w = img.shape; sx, sy = map(int, map(round, seed_xy))
    x0, x1 = max(0, sx-half_w), min(w, sx+half_w+1); y0, y1 = max(0, sy-half_h), min(h, sy+half_h+1)
    roi = img[y0:y1, x0:x1].astype(np.float32)
    grad = np.abs(cv2.Sobel(roi, cv2.CV_32F, 1, 0, ksize=3))
    sy0, sx0 = np.clip(sy-y0, 0, roi.shape[0]-1), np.clip(sx-x0, 0, roi.shape[1]-1)
    edge = np.full(roi.shape[0], np.nan, dtype=float)

    left, right = max(0, sx0-2*continuity), min(roi.shape[1], sx0+2*continuity+1)
    start = left + int(np.argmax(grad[int(sy0), left:right])); edge[int(sy0)] = start

    for direction in (1, -1):
        prev = start
        for yy in range(int(sy0)+direction, roi.shape[0] if direction > 0 else -1, direction):
            lo, hi = max(0, int(prev)-continuity), min(roi.shape[1], int(prev)+continuity+1)
            xx = lo + int(np.argmax(grad[yy, lo:hi])); edge[yy] = xx; prev = xx

    ys = np.where(np.isfinite(edge))[0]; xs = edge[np.isfinite(edge)]
    if len(xs) < 9: return None
    win = min(31, len(xs) if len(xs) % 2 else len(xs)-1)
    xs = savgol_filter(xs, max(win, 7), 2, mode="interp")
    return np.column_stack([xs + x0, ys + y0])

def turning_angles(points, window=8):
    out = np.full(len(points), np.nan)
    for i in range(window, len(points)-window):
        v1, v2 = points[i] - points[i-window], points[i+window] - points[i]
        if np.linalg.norm(v1) == 0 or np.linalg.norm(v2) == 0: continue
        v1, v2 = v1 / np.linalg.norm(v1), v2 / np.linalg.norm(v2)
        out[i] = np.degrees(np.arctan2(v1[0]*v2[1]-v1[1]*v2[0], np.clip(v1 @ v2, -1, 1)))
    return out

def analyze_hip_bump(path, bump_point):
    _, img, _, _ = preprocess_dicom(path)  # без threshold
    edge = trace_edge(img, bump_point)
    if edge is None: return None
    angles = turning_angles(edge, max(5, len(edge)//35))
    idx = int(np.nanargmax(np.abs(angles)))
    fig, ax = plt.subplots(figsize=FIG_TALL); ax.imshow(img, cmap="gray"); ax.plot(edge[:,0], edge[:,1], lw=1.5); ax.scatter(edge[idx,0], edge[idx,1], s=25)
    ax.set_title(f"Горбик: {angles[idx]:.1f}°"); ax.axis("off"); plt.tight_layout(); plt.show()
    return {"turning_angle_deg": float(angles[idx]), "point": tuple(edge[idx])}

## 7. Бедро — металл: отличие скрытого представления

In [ ]:
_EMBED_ENCODER = None

def get_embedding_encoder():
    global _EMBED_ENCODER
    if _EMBED_ENCODER is None:
        model = resnet18(weights=ResNet18_Weights.DEFAULT).to(DEVICE).eval()
        _EMBED_ENCODER = nn.Sequential(*list(model.children())[:-1]).eval()
    return _EMBED_ENCODER

@torch.no_grad()
def pretrained_embedding(path):
    encoder = get_embedding_encoder()
    x = transform(dicom_to_pil(path)).unsqueeze(0).to(DEVICE)
    return encoder(x).flatten(1)[0].cpu().numpy()


def load_metal_table():
    """Использует колонку metal из единого Размеченные/labels.csv."""
    legs = labels_df[labels_df["label"].eq("LEG")].copy()
    if "metal" not in legs.columns:
        print("В labels.csv пока нет колонки metal. Обновите Docker labeler и разметьте металл.")
        return pd.DataFrame()

    legs["metal"] = legs["metal"].astype(str).str.strip()
    out = legs[legs["metal"].isin(["0", "1"])].copy()
    counts = out["metal"].value_counts().to_dict()
    print(f"Металл размечен: {len(out)} / {len(legs)} ног. Распределение: {counts}")
    return out


def metal_embedding_demo(n_normals=40):
    m = load_metal_table()
    if m.empty:
        print("Сначала разметьте metal=0/1 через Docker-интерфейс, затем перезапустите ячейку загрузки labels_df.")
        return pd.DataFrame()
    if not set(m["metal"]) >= {"0", "1"}:
        print("Для демонстрации нужны оба класса: хотя бы один metal=0 и один metal=1.")
        return pd.DataFrame()

    normals = m[m["metal"].eq("0")].sample(min(n_normals, m["metal"].eq("0").sum()), random_state=SEED)
    metal = m[m["metal"].eq("1")].iloc[[0]]
    sample = pd.concat([normals, metal], ignore_index=True)

    emb = np.vstack([pretrained_embedding(p) for p in sample["dicom_path"]])
    centroid = emb[:-1].mean(0)
    dist = np.linalg.norm(emb - centroid, axis=1)
    xy = PCA(2, random_state=SEED).fit_transform(emb)

    fig, ax = plt.subplots(figsize=FIG_SMALL)
    ax.scatter(xy[:-1, 0], xy[:-1, 1], s=14, label="no metal")
    ax.scatter(xy[-1, 0], xy[-1, 1], s=45, marker="x", label="metal")
    ax.set_title("Pretrained ResNet embeddings")
    ax.legend(fontsize=7)
    plt.tight_layout(); plt.show()

    return pd.DataFrame({
        "metal": sample["metal"],
        "distance_to_normal_centroid": dist,
        "relative_path": sample["relative_path"],
    }).sort_values("distance_to_normal_centroid", ascending=False)

## 8. Позвоночник — ось по 10% самых ярких пикселей и кусочно-линейная аппроксимация

Новая схема:

1. Во всём изображении оставляются ровно `10%` самых ярких пикселей.
2. В центральной области позвоночника для каждой горизонтальной строки считается центр масс этих ярких пикселей.
3. Последовательность центров масс аппроксимируется **7 линейными участками** — это оси 7 позвонков.
4. Между соседними точками разрыва производной требуется расстояние не меньше `H / 7`, где `H` — высота исходного изображения.
5. Перпендикуляры строятся уже к кусочно-линейной оси; по ним считается суммарная яркость и ищутся межпозвонковые антипики.
6. Угол позвоночника — угол линии от верхней точки первого сегмента до нижней точки последнего с вертикалью.
7. Прототип показателя сколиоза — модуль разности углов первого и последнего линейных сегментов.

In [ ]:
def crop_spine_roi(img, x_range=(0.20, 0.80), y_range=(0.04, 0.96)):
    h, w = img.shape
    x0, x1 = int(w*x_range[0]), int(w*x_range[1])
    y0, y1 = int(h*y_range[0]), int(h*y_range[1])
    return img[y0:y1, x0:x1], (x0, y0, x1, y1)


def top_fraction_mask(img, fraction=SPINE_BRIGHT_FRACTION):
    """Ровно fraction самых ярких пикселей всего изображения."""
    flat = np.asarray(img, dtype=np.float32).ravel()
    n_keep = max(1, int(round(len(flat) * fraction)))
    idx = np.argpartition(flat, -n_keep)[-n_keep:]
    mask = np.zeros(len(flat), dtype=bool)
    mask[idx] = True
    return mask.reshape(img.shape)


def row_centers_of_mass(roi, bright_mask_roi):
    weights = roi * bright_mask_roi.astype(np.float32)
    row_mass = weights.sum(axis=1)
    xs = np.arange(roi.shape[1], dtype=np.float32)
    xcm = np.full(roi.shape[0], np.nan, dtype=np.float32)
    valid = row_mass > 0
    xcm[valid] = (weights[valid] * xs[None, :]).sum(axis=1) / row_mass[valid]

    # Пропуски нужны только для устойчивости fit; они получают очень маленький вес.
    if valid.sum() < 2:
        xcm[:] = roi.shape[1] / 2
    else:
        xcm = np.interp(np.arange(len(xcm)), np.where(valid)[0], xcm[valid]).astype(np.float32)

    confidence = row_mass / max(float(row_mass.max()), 1e-6)
    confidence = np.where(valid, np.maximum(confidence, 0.05), 0.01).astype(np.float64)
    return xcm.astype(np.float64), row_mass.astype(np.float64), confidence


def regression_prefix(y, x, w):
    values = {
        "w": w,
        "wy": w*y,
        "wx": w*x,
        "wyy": w*y*y,
        "wyx": w*y*x,
        "wxx": w*x*x,
    }
    return {k: np.r_[0.0, np.cumsum(v)] for k, v in values.items()}


def interval_line_fit(prefix, i, j):
    """Weighted x = intercept + slope*y on [i, j). Returns SSE, intercept, slope."""
    def s(name):
        return prefix[name][j] - prefix[name][i]

    sw, sy, sx = s("w"), s("wy"), s("wx")
    syy, syx, sxx = s("wyy"), s("wyx"), s("wxx")
    if sw <= 1e-8 or j - i < 2:
        return np.inf, 0.0, 0.0

    det = sw*syy - sy*sy
    if abs(det) < 1e-10:
        slope = 0.0
        intercept = sx / sw
    else:
        slope = (sw*syx - sy*sx) / det
        intercept = (sx - slope*sy) / sw

    sse = sxx - intercept*sx - slope*syx
    return max(float(sse), 0.0), float(intercept), float(slope)


def fit_piecewise_spine_axis(xcm, confidence, full_image_height, n_breaks=6):
    """
    Глобальный piecewise-linear fit с 6 разрывами производной.
    Ограничение относится к соседним breakpoints: >= full_image_height/7.
    """
    h = len(xcm)
    y = np.arange(h, dtype=np.float64)
    prefix = regression_prefix(y, xcm, confidence)

    min_gap = max(2, int(math.ceil(full_image_height / SPINE_MIN_SEPARATOR_DISTANCE_DIVISOR)))
    edge_min = max(3, int(round(0.02*h)))
    candidate_step = 2 if h > 500 else 1

    # dp[(k, start)] = (cost, [breakpoints...]), где k — сколько разрывов осталось.
    from functools import lru_cache

    @lru_cache(None)
    def solve(k, start):
        if k == 0:
            cost, _, _ = interval_line_fit(prefix, start, h)
            return cost, ()

        first_min_len = edge_min if start == 0 else min_gap
        lo = start + first_min_len
        # После b нужно разместить k-1 разрывов с min_gap и оставить небольшой хвост.
        hi = h - edge_min - (k - 1)*min_gap
        if lo > hi:
            return np.inf, ()

        best_cost, best_breaks = np.inf, ()
        candidates = list(range(lo, hi + 1, candidate_step))
        if hi not in candidates:
            candidates.append(hi)

        for b in candidates:
            left_cost, _, _ = interval_line_fit(prefix, start, b)
            if not np.isfinite(left_cost):
                continue
            right_cost, right_breaks = solve(k - 1, b)
            total = left_cost + right_cost
            if total < best_cost:
                best_cost = total
                best_breaks = (b,) + right_breaks
        return best_cost, best_breaks

    total_cost, breaks = solve(n_breaks, 0)
    breaks = np.array(breaks, dtype=int)
    if len(breaks) != n_breaks:
        raise RuntimeError(
            f"Не удалось разместить {n_breaks} разрывов с min_gap={min_gap}px. "
            f"ROI height={h}, full height={full_image_height}."
        )

    bounds = np.r_[0, breaks, h]
    fit_x = np.zeros(h, dtype=np.float64)
    segments = []

    for i, (a, b) in enumerate(zip(bounds[:-1], bounds[1:])):
        _, intercept, slope = interval_line_fit(prefix, int(a), int(b))
        yy = np.arange(a, b, dtype=np.float64)
        fit_x[a:b] = intercept + slope*yy
        segments.append({
            "segment_index": i,
            "segment_name": SPINE_SEGMENT_NAMES[i] if i < len(SPINE_SEGMENT_NAMES) else f"segment_{i}",
            "y_top": int(a),
            "y_bottom": int(b-1),
            "intercept": intercept,
            "slope_dx_dy": slope,
            "vertical_angle": math.degrees(math.atan(slope)),
        })

    return fit_x, breaks, pd.DataFrame(segments), min_gap, total_cost


def piecewise_local_slopes(segments, h):
    slopes = np.zeros(h, dtype=np.float64)
    for _, row in segments.iterrows():
        a, b = int(row["y_top"]), int(row["y_bottom"]) + 1
        slopes[a:b] = float(row["slope_dx_dy"])
    return slopes


def perpendicular_profile(roi, axis_x, local_slopes, half_length_frac=0.42):
    h, w = roi.shape
    half = int(w * half_length_frac)
    t = np.arange(-half, half + 1, dtype=np.float64)
    profile = np.zeros(h, dtype=np.float64)
    lines = []

    for y in range(h):
        slope = local_slopes[y]
        normal = np.array([1.0, -slope], dtype=np.float64)
        normal /= max(np.linalg.norm(normal), 1e-8)
        x = axis_x[y] + t*normal[0]
        yy = y + t*normal[1]
        vals = map_coordinates(roi, [yy, x], order=1, mode="constant", cval=0)
        profile[y] = vals.sum()
        lines.append((np.array([axis_x[y], float(y)]), normal))

    smooth = gaussian_filter1d(profile, sigma=max(1.2, h/200))
    return profile, smooth, lines


def select_separator_antipeaks(profile, full_image_height):
    min_distance = max(2, int(math.ceil(full_image_height / SPINE_MIN_SEPARATOR_DISTANCE_DIVISOR)))
    peaks, props = find_peaks(
        -profile,
        distance=min_distance,
        prominence=max(float(np.std(profile))*0.08, 1e-6),
    )
    if len(peaks) > SPINE_N_SEPARATORS:
        keep = np.argsort(props["prominences"])[-SPINE_N_SEPARATORS:]
        peaks = peaks[keep]
    return np.sort(peaks.astype(int)), min_distance


def analyze_spine_geometry(path, show=True):
    _, img, filtered, _ = preprocess_dicom(path)

    # 1) top 10% определяется по всему изображению.
    bright_mask = top_fraction_mask(img, SPINE_BRIGHT_FRACTION)

    # 2) затем анализируем центральную область, сохраняя тот же global mask.
    roi, (x0, y0, x1, y1) = crop_spine_roi(filtered)
    bright_roi = bright_mask[y0:y1, x0:x1]
    xcm, row_mass, confidence = row_centers_of_mass(roi, bright_roi)

    # 3) 7 линейных осей / 6 разрывов производной.
    axis_x, derivative_breaks, segments, break_min_gap, fit_cost = fit_piecewise_spine_axis(
        xcm, confidence, full_image_height=img.shape[0], n_breaks=6
    )
    slopes = piecewise_local_slopes(segments, len(axis_x))

    # 4) суммы яркости по перпендикулярам и межпозвонковые антипики.
    _, perp_sum, perp_lines = perpendicular_profile(roi, axis_x, slopes)
    separators, separator_min_gap = select_separator_antipeaks(perp_sum, img.shape[0])

    # 5) углы.
    first = segments.iloc[0]
    last = segments.iloc[-1]
    top = np.array([axis_x[0], 0.0])
    bottom = np.array([axis_x[-1], float(len(axis_x)-1)])
    spine_angle = math.degrees(math.atan2(bottom[0]-top[0], bottom[1]-top[1]))
    scoliosis_proxy = abs(float(first["vertical_angle"] - last["vertical_angle"]))

    if show:
        fig, axes = plt.subplots(1, 3, figsize=(8.5, 3.2))

        # Геометрия.
        axes[0].imshow(roi, cmap="gray")
        axes[0].scatter(xcm, np.arange(len(xcm)), s=1.5, alpha=0.35, label="row COM")
        for _, seg in segments.iterrows():
            a, b = int(seg["y_top"]), int(seg["y_bottom"])+1
            axes[0].plot(axis_x[a:b], np.arange(a, b), lw=1.6)
        for y in derivative_breaks:
            axes[0].axhline(y, lw=.6, ls=":", alpha=.6)
        for y in separators:
            c, n = perp_lines[y]
            tt = np.array([-roi.shape[1]*0.42, roi.shape[1]*0.42])
            pts = c[:, None] + n[:, None]*tt
            axes[0].plot(pts[0], pts[1], lw=1.1)
        axes[0].plot([top[0], bottom[0]], [top[1], bottom[1]], "--", lw=1.0)
        axes[0].set_title(f"axis={spine_angle:.1f}°, Δ1-7={scoliosis_proxy:.1f}°", fontsize=8)
        axes[0].axis("off")

        # Что именно осталось после top 10%.
        axes[1].imshow(roi, cmap="gray")
        axes[1].imshow(np.ma.masked_where(~bright_roi, bright_roi), cmap="autumn", alpha=.45)
        axes[1].plot(axis_x, np.arange(len(axis_x)), lw=1.2)
        axes[1].set_title("Top 10% + piecewise axis", fontsize=8)
        axes[1].axis("off")

        # Перпендикулярный профиль.
        axes[2].plot(perp_sum, np.arange(len(perp_sum)))
        if len(separators):
            axes[2].scatter(perp_sum[separators], separators, s=16)
        axes[2].invert_yaxis()
        axes[2].set_title(f"Perpendicular sums\nmin gap={separator_min_gap}px", fontsize=8)
        axes[2].tick_params(labelsize=6)

        plt.tight_layout(); plt.show()

    return {
        "spine_axis_angle_deg": float(spine_angle),
        "scoliosis_proxy_deg": float(scoliosis_proxy),
        "axis_x": axis_x,
        "row_centers_x": xcm,
        "segments": segments,
        "derivative_breaks": derivative_breaks,
        "separators": separators,
        "perpendicular_profile": perp_sum,
        "break_min_gap_px": break_min_gap,
        "separator_min_gap_px": separator_min_gap,
        "piecewise_fit_cost": float(fit_cost),
    }

Ключевое изменение: оси позвонков теперь **не ищутся отдельной процедурой симметрии внутри полос**. Семь линейных участков piecewise-fit сами трактуются как оси семи позвонков. Шесть разрывов их производной ограничены по расстоянию. Уже к этим осям строятся перпендикуляры для профиля межпозвонковых разделений.

`spine_axis_angle_deg` — угол общей верхняя→нижняя ось с вертикалью.  
`scoliosis_proxy_deg` — разность углов первого и последнего линейных участков; это исследовательский proxy, **не клинический Cobb angle**.

## 9. Позвоночник — позиция: pretrained SAM для подвздошных костей

In [ ]:
_SAM_PREDICTOR = None

def ensure_sam_checkpoint():
    if SAM_CHECKPOINT.exists(): return SAM_CHECKPOINT
    print("Скачиваю pretrained SAM ViT-B (~375 MB)...")
    urllib.request.urlretrieve(SAM_CHECKPOINT_URL, SAM_CHECKPOINT)
    return SAM_CHECKPOINT

def get_sam_predictor():
    global _SAM_PREDICTOR
    if _SAM_PREDICTOR is not None: return _SAM_PREDICTOR
    try:
        from segment_anything import sam_model_registry, SamPredictor
    except ImportError as e:
        raise ImportError("Установите Segment Anything: %pip install -q https://github.com/facebookresearch/segment-anything/archive/refs/heads/main.zip") from e
    sam = sam_model_registry[SAM_MODEL_TYPE](checkpoint=str(ensure_sam_checkpoint())).to(DEVICE)
    _SAM_PREDICTOR = SamPredictor(sam)
    return _SAM_PREDICTOR

def weighted_prompt(img, x0, x1, y0, y1):
    crop = img[y0:y1, x0:x1]; yy, xx = np.mgrid[y0:y1, x0:x1]
    weights = np.maximum(crop - np.quantile(crop, 0.65), 0)
    if weights.sum() == 0: return ((x0+x1)/2, (y0+y1)/2)
    return (float((xx*weights).sum()/weights.sum()), float((yy*weights).sum()/weights.sum()))

def segment_iliac_bones(path, show=True):
    _, img, _, _ = preprocess_dicom(path)
    h, w = img.shape
    rgb = np.repeat((img*255).astype(np.uint8)[...,None], 3, axis=2)
    predictor = get_sam_predictor(); predictor.set_image(rgb)

    regions = {"left_image": (int(.03*w), int(.48*w), int(.55*h), int(.98*h)),
               "right_image": (int(.52*w), int(.97*w), int(.55*h), int(.98*h))}
    masks_out, rows = {}, []

    for name, (x0, x1, y0, y1) in regions.items():
        point = weighted_prompt(img, x0, x1, y0, y1)
        masks, scores, _ = predictor.predict(point_coords=np.array([point]), point_labels=np.array([1]), multimask_output=True)

        zone = np.zeros((h, w), bool); zone[y0:y1, x0:x1] = True
        best = None
        for mask, sam_score in zip(masks, scores):
            area = mask.mean()
            overlap = (mask & zone).sum() / max(mask.sum(), 1)
            candidate_score = float(sam_score) + 0.6*overlap - 0.5*max(0, area-0.25)
            if 0.005 <= area <= 0.35 and (best is None or candidate_score > best[0]):
                best = (candidate_score, mask, float(sam_score), point)
        if best is None:
            i = int(np.argmax(scores)); best = (float(scores[i]), masks[i], float(scores[i]), point)

        masks_out[name] = best[1]
        rows.append({"side_in_image": name, "sam_score": best[2], "area_fraction": float(best[1].mean()), "prompt_x": point[0], "prompt_y": point[1]})

    union = masks_out["left_image"] | masks_out["right_image"]
    total_fraction = float(union.mean())

    if show:
        fig, ax = plt.subplots(figsize=FIG_TALL); ax.imshow(img, cmap="gray")
        overlay = np.ma.masked_where(~union, union)
        ax.imshow(overlay, alpha=.35, cmap="autumn")
        ax.set_title(f"Подвздошные сегменты: {100*total_fraction:.1f}% площади"); ax.axis("off")
        plt.tight_layout(); plt.show()

    return total_fraction, pd.DataFrame(rows), masks_out

SAM здесь используется как **generic pretrained segmentation model**. Ему не передаётся готовая маска. Точки-подсказки автоматически выбираются в нижней левой и нижней правой областях по яркости, после чего среди SAM-масок выбирается наиболее согласованная с ожидаемым расположением подвздошных костей.

Это прототип: качество выбора сегмента необходимо проверить на реальных DXA.

## 10. Шесть демонстраций

Вместо закомментированных вызовов используются переключатели:

- `RUN_SPINE_DEMOS` — автоматические визуализации позвоночника;
- `RUN_SAM_DEMO` — подвздошные кости; отдельно, потому что при первом запуске скачивается SAM checkpoint;
- `RUN_METAL_DEMO` — ResNet embeddings;
- `RUN_INTERACTIVE_HIP_DEMOS` — позиция и ротация бедра; отдельно, потому что требуют кликов мышью.

Если `LEFT/RIGHT` ещё не размечены, для демонстрации бедра временно берутся обычные изображения `LEG`, поэтому `.iloc[0]` больше не падает.

In [ ]:
def path_for_study(study, label=None, side=None):
    q = labels_df[labels_df["study_id"].eq(str(study))]
    if label is not None:
        q = q[q["label"].eq(label)]
    if side is not None:
        q = q[q["side"].eq(side)]
    return q.iloc[0]["dicom_path"] if len(q) else None


def first_path(mask, description):
    paths = labels_df.loc[mask, "dicom_path"]
    if len(paths):
        return paths.iloc[0]
    print(f"Нет размеченного изображения для: {description}")
    return None


def get_demo_paths():
    spine = first_path(labels_df["label"].eq("SPINE"), "SPINE")
    left = first_path(labels_df["anatomy3"].eq("LEG_LEFT"), "LEG_LEFT")
    right = first_path(labels_df["anatomy3"].eq("LEG_RIGHT"), "LEG_RIGHT")

    # Пока side ещё не заполнен, demo бедра всё равно можно запускать на обычных LEG.
    legs = labels_df.loc[labels_df["label"].eq("LEG"), "dicom_path"].tolist()
    if left is None and legs:
        left = legs[0]
        print("LEFT ещё не размечен — для demo используется обычный LEG.")
    if right is None and legs:
        right = legs[1] if len(legs) > 1 else legs[0]
        print("RIGHT ещё не размечен — для demo используется другой LEG.")
    return {"spine": spine, "left_leg": left, "right_leg": right}


def demo_1_hip_position(path, points=None):
    print("DEMO 1 — бедро / позиция")
    if path is None:
        print("Нет доступного изображения бедра.")
        return None
    if points is None:
        print("Кликните три опорные точки 1 → 2 → 3.")
        points = select_points(path)
    return analyze_hip_position(path, points)


def demo_2_hip_rotation(path, bump_point=None):
    print("DEMO 2 — бедро / ротация")
    if path is None:
        print("Нет доступного изображения бедра.")
        return None
    if bump_point is None:
        print("Кликните примерную точку горбика.")
        bump_point = select_points(path, ("bump",))["bump"]
    return analyze_hip_bump(path, bump_point)


def demo_3_hip_metal():
    print("DEMO 3 — бедро / металл")
    return metal_embedding_demo()


def demo_4_spine_angle(path):
    print("DEMO 4 — позвоночник / угол с вертикалью")
    if path is None:
        return None
    result = analyze_spine_geometry(path, show=True)
    return {"spine_axis_angle_deg": result["spine_axis_angle_deg"]}


def demo_5_spine_position(path):
    print("DEMO 5 — позвоночник / подвздошные кости")
    if path is None:
        return None
    fraction, table, _ = segment_iliac_bones(path, show=True)
    print(f"Суммарная доля площади: {100*fraction:.2f}%")
    return table


def demo_6_spine_scoliosis(path):
    print("DEMO 6 — позвоночник / проблемы (proxy сколиоза)")
    if path is None:
        return None
    result = analyze_spine_geometry(path, show=True)
    return {"first_last_axis_difference_deg": result["scoliosis_proxy_deg"]}


demo_paths = get_demo_paths()
spine_demo_path = demo_paths["spine"]
left_leg_path = demo_paths["left_leg"]
right_leg_path = demo_paths["right_leg"]

print("\nDemo paths:")
print("SPINE:", spine_demo_path)
print("LEFT/LEG demo:", left_leg_path)
print("RIGHT/LEG demo:", right_leg_path)

# Автоматические графики можно запускать через Run All.
RUN_SPINE_DEMOS = True

# SAM при первом запуске скачивает большой checkpoint (~375 MB).
RUN_SAM_DEMO = False

# Работает после разметки колонки metal=0/1 в основном labels.csv.
RUN_METAL_DEMO = True

# Эти две демонстрации требуют кликов мышью; поэтому по умолчанию выключены.
RUN_INTERACTIVE_HIP_DEMOS = False

if RUN_SPINE_DEMOS:
    demo_4_spine_angle(spine_demo_path)
    demo_6_spine_scoliosis(spine_demo_path)

if RUN_SAM_DEMO:
    display(demo_5_spine_position(spine_demo_path))

if RUN_METAL_DEMO:
    display(demo_3_hip_metal())

if RUN_INTERACTIVE_HIP_DEMOS:
    display(demo_1_hip_position(left_leg_path))
    demo_2_hip_rotation(left_leg_path)

## 11. Использование `разметка.csv` для выбора демонстрационных случаев

In [ ]:
def studies_with_task_value(task, value="1"):
    if task not in task_labels.columns:
        return []
    values = task_labels[task].astype(str).str.strip()
    return task_labels.loc[values.eq(str(value)), "study"].tolist()


def scoliosis_studies():
    if "scoliosis_from_comment" not in task_labels.columns:
        return []
    return task_labels.loc[task_labels["scoliosis_from_comment"].eq(1), "study"].tolist()


print("Строк в task_labels:", len(task_labels))

if task_labels.empty:
    print(
        "task_labels пуст. Проверь предыдущую ячейку: "
        "какой файл разметки найден и распознаны ли study UID."
    )
else:
    checks = {
        "spine_alignment=1": studies_with_task_value("spine_alignment", "1"),
        "scoliosis in comment": scoliosis_studies(),
        "right hip position/rotation=1": studies_with_task_value("right_hip_position_rotation", "1"),
        "left hip position/rotation=1": studies_with_task_value("left_hip_position_rotation", "1"),
    }
    for name, studies in checks.items():
        print(f"{name}: n={len(studies)}, examples={studies[:10]}")

## Что удалено по сравнению с v4

- merge/recovery нескольких `labels.csv`;
- отдельные старые функции для горизонтального профиля позвоночника;
- заранее заданные маски подвздошных костей и U-Net-заглушка;
- дублирующие train/val helpers;
- крупные диагностические фигуры;
- лишние промежуточные визуализации.

Основной код теперь ориентирован на один финальный датасет и шесть целевых проверок.